# PDF Ingestion
- Ingest PDF reports and extract clean text using pdfplumber
- Made because the initial training data on txt files made from CSV was not semantic enough, it was too structured and we need more unstructured data for semantic analysis

## Setup

In [32]:
# !pip install -r ../requirements.txt

In [33]:
# Imports
import pdfplumber
from pathlib import Path
import re

In [34]:
# Paths
PROJECT_ROOT    = Path().resolve().parent
PDF_DIR         = PROJECT_ROOT / "data" / "1_source" / "PDF Reports"
CLEANED_DIR     = PROJECT_ROOT / "data" / "2_cleaned" / "PDF_Reports"
KB_REPORTS_DIR  = PROJECT_ROOT / "data" / "3_txt_KB" / "PDF_Reports"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

pdf_files = [
    "FRA_2025.pdf",
    "ILGA_2023.pdf",
    "ILGA_2024.pdf",
    "ILGA_2025.pdf"
]

## Processing Files
### Helper functions

In [35]:
def clean_text(text: str) -> str:
    # Remove excessive whitespace and line breaks
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)

    # Remove page numbers (simple heuristic)
    text = re.sub(r'\n\d+\n', '\n', text)

    # Remove hyphenation across lines
    text = re.sub(r'-\n', '', text)

    return text.strip()

In [36]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    full_text = []

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            # Extract only text (ignores images/graphs automatically)
            text = page.extract_text()
            if text:
                full_text.append(text)

    return "\n".join(full_text)

## Apply cleaning
- convert PDF to text
- output is in `data/2_cleaned/`

In [37]:
# for pdf_name in pdf_files:
#     pdf_path = PDF_DIR / pdf_name

#     print(f"Processing: {pdf_name}")

#     raw_text = extract_text_from_pdf(pdf_path)
#     cleaned_text = clean_text(raw_text)

#     output_file = CLEANED_DIR / f"{pdf_name.replace('.pdf', '.txt')}"

#     with open(output_file, "w", encoding="utf-8") as f:
#         f.write(cleaned_text)

#     print(f"Saved to: {output_file}\n")

## Note
- Commented out execution as the outputted files have been reviewed and processed, so don't want to overwrite them

# Chunk txt files

In [38]:
files = [
    "FRA_2025.txt",
    "ILGA_2023.txt",
    "ILGA_2024.txt",
    "ILGA_2025.txt"
]

In [39]:
# Helper function
def write_section(file_prefix, section_name, content, metadata):
    safe_name = re.sub(r'[^A-Za-z0-9]+', '_', section_name).strip('_')
    filename = f"{file_prefix}__{safe_name}.txt"
    output_path = KB_REPORTS_DIR / filename

    with open(output_path, "w", encoding="utf-8") as f:
        for k, v in metadata.items():
            f.write(f"{k}: {v}\n")
        f.write("\n")
        f.write(content.strip())

### ILGA Reports

In [40]:
COUNTRIES = [
    "ALBANIA","ANDORRA","ARMENIA","AUSTRIA","AZERBAIJAN","BELARUS","BELGIUM",
    "BOSNIA AND HERZEGOVINA","BULGARIA","CROATIA","CYPRUS","CZECHIA","DENMARK",
    "ESTONIA","FINLAND","FRANCE","GEORGIA","GERMANY","GREECE","HUNGARY",
    "ICELAND","IRELAND","ITALY","KAZAKHSTAN","KOSOVO","KYRGYZSTAN","LATVIA",
    "LIECHTENSTEIN","LITHUANIA","LUXEMBOURG","MALTA","MOLDOVA","MONACO",
    "MONTENEGRO","NETHERLANDS","NORTH MACEDONIA","NORWAY","POLAND","PORTUGAL",
    "ROMANIA","RUSSIA","SAN MARINO","SERBIA","SLOVAKIA","SLOVENIA","SPAIN",
    "SWEDEN","SWITZERLAND","TAJIKISTAN","TURKEY","TURKMENISTAN","UKRAINE",
    "UNITED KINGDOM","UZBEKISTAN"
]

INSTITUTIONS = [
    "EUROPEAN UNION",
    "COUNCIL OF EUROPE",
    "UNITED NATIONS",
    "ORGANISATION FOR SECURITY AND COOPERATION IN EUROPE"
]

In [41]:
def split_ilga(text, year):
    sections = {}

    # Build pattern
    headers = COUNTRIES + INSTITUTIONS
    pattern = r"\n(" + "|".join(re.escape(h) for h in headers) + r")\n"

    splits = re.split(pattern, text)

    # Structure: [intro, HEADER, content, HEADER, content...]
    for i in range(1, len(splits), 2):
        header = splits[i]
        content = splits[i + 1]

        sections[header] = content

    return sections



### FRA Reports

In [42]:
def split_fra(text):
    lines = text.split("\n")

    sections = {}
    current_key = None

    section_pattern = re.compile(r"^(\d+)\.\s+(.*)")
    subsection_pattern = re.compile(r"^(\d+\.\d+)\.\s+(.*)")
    deep_pattern = re.compile(r"^(\d+\.\d+\.\d+)")

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # SECTION (e.g., 5.)
        sec_match = section_pattern.match(line)
        sub_match = subsection_pattern.match(line)
        deep_match = deep_pattern.match(line)

        if sec_match:
            current_key = f"{sec_match.group(1)}. {sec_match.group(2)}"
            sections[current_key] = []

        elif sub_match:
            current_key = f"{sub_match.group(1)}. {sub_match.group(2)}"
            sections[current_key] = []

        elif deep_match:
            # Ignore creating new section → append to current
            if current_key:
                sections[current_key].append(line)

        else:
            if current_key:
                sections[current_key].append(line)

    # Join content
    sections = {
        k: "\n".join(v).strip()
        for k, v in sections.items()
        if v
    }

    return sections

### Main function
- Splits the cleaned .txt files into sections, using the report structure to guide this. 
- Adds metadata for better retrieval

In [43]:
for file in files:
    path = CLEANED_DIR / file

    if not path.exists():
        print(f"Missing: {file}")
        continue

    text = path.read_text(encoding="utf-8")

    print(f"\nProcessing: {file}")

    # FRA
    if "FRA" in file:
        sections = split_fra(text)

        for section, content in sections.items():
            metadata = {
                "SOURCE": "FRA",
                "YEAR": "2025",
                "SECTION_TYPE": "REPORT_SECTION",
                "SECTION": section
            }

            write_section("FRA_2025", section, content, metadata)

    # ILGA
    elif "ILGA" in file:
        year = re.search(r"\d{4}", file).group()
        sections = split_ilga(text, year)

        for section, content in sections.items():
            section_type = "COUNTRY" if section in COUNTRIES else "INSTITUTION"

            metadata = {
                "SOURCE": "ILGA",
                "YEAR": year,
                "SECTION_TYPE": section_type,
                "SECTION": section
            }

            write_section(f"ILGA_{year}", section, content, metadata)

print("\nDone.")


Processing: FRA_2025.txt

Processing: ILGA_2023.txt

Processing: ILGA_2024.txt

Processing: ILGA_2025.txt

Done.
